<a href="https://colab.research.google.com/github/shin584/project/blob/3D_simulation/DNABERT_Data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

def prepare_data_split(file_path="SaCas9_v4.xlsx"):
    print("=== [Phase 1.1] 원본 데이터 로드 및 유효성 검사 ===")
    # 1. 파일 로드 (Excel 및 CSV 동시 지원)
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"'{file_path}' 파일을 찾을 수 없습니다.")

    if file_path.endswith('.xlsx'):
        df = pd.read_excel(file_path, engine='openpyxl')
    else:
        df = pd.read_csv(file_path)

    print(f"원본 데이터 크기: {df.shape}")

    # 2. 타겟 컬럼 동적 추출 (기존 main.py 유연성 확보 로직 차용)
    seq_col = next((c for c in df.columns if c.lower() in ["input_sequence", "sequence", "target_sequence", "seq"]), df.columns[0])
    label_col = next((c for c in df.columns if c.lower() in ["score", "label", "efficiency", "cleavage_efficiency", "cleavage_score"]), df.columns[1])

    print(f"인식된 서열 컬럼: '{seq_col}'")
    print(f"인식된 라벨 컬럼: '{label_col}'")

    # 3. 분석에 필요한 컬럼만 추출 및 결측치(NaN) 제거
    df = df[[seq_col, label_col]].copy()
    initial_len = len(df)
    df = df.dropna()
    print(f"결측치 제거: {initial_len - len(df)}개 행 삭제됨 (현재 크기: {df.shape})")

    print("\n=== [Phase 1.2] 난수 고정 및 셔플링 ===")
    SEED = 42
    np.random.seed(SEED)

    # 전체 셔플링
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"random_state={SEED} 로 전체 데이터 셔플링 완료.")

    print("\n=== [Phase 1.3] 8:1:1 비율 3분할 ===")
    # 1차 분할: 전체 데이터의 80%를 Train, 20%를 Temp로 분할
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)

    # 2차 분할: Temp(20%)의 절반씩 분할하여 Val(10%), Test(10%) 생성
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)

    print("\n=== [Phase 1.4] 독립 파일 저장 ===")
    train_df.to_csv("train.csv", index=False)
    val_df.to_csv("val.csv", index=False)
    test_df.to_csv("test.csv", index=False)
    print("파일 저장 완료: 'train.csv', 'val.csv', 'test.csv' (인덱스 제외)")

    print("\n=== [Phase 1.5] 분할 무결성 검증 (Integrity Check) ===")
    total = len(df)
    print(f"전체 유효 데이터 개수: {total}")
    print(f"Train Set: {len(train_df)}개 ({len(train_df)/total*100:.1f}%)")
    print(f"Val Set  : {len(val_df)}개 ({len(val_df)/total*100:.1f}%)")
    print(f"Test Set : {len(test_df)}개 ({len(test_df)/total*100:.1f}%)")

    # 교집합 검사 (Data Leakage 방지: Test 서열이 Train에 포함되었는지 확인)
    train_seqs = set(train_df[seq_col])
    test_seqs = set(test_df[seq_col])
    leakage = train_seqs.intersection(test_seqs)

    if len(leakage) == 0:
        print("무결성 검증 통과: Train과 Test 세트 간 중복 서열이 완벽하게 격리되었습니다.")
    else:
        print(f"경고: {len(leakage)}개의 중복 서열이 발견되었습니다! (동일한 서열이 여러 개의 라벨을 가지고 있는지 원본 데이터를 확인하세요)")

if __name__ == "__main__":
    # 작업하시는 디렉토리에 'SaCas9_v4.xlsx' 파일이 동일한 위치에 있어야 합니다.
    prepare_data_split(file_path="SaCas9_v4.xlsx")

=== [Phase 1.1] 원본 데이터 로드 및 유효성 검사 ===
원본 데이터 크기: (5145, 4)
인식된 서열 컬럼: 'input_sequence'
인식된 라벨 컬럼: 'score'
결측치 제거: 0개 행 삭제됨 (현재 크기: (5145, 2))

=== [Phase 1.2] 난수 고정 및 셔플링 ===
random_state=42 로 전체 데이터 셔플링 완료.

=== [Phase 1.3] 8:1:1 비율 3분할 ===

=== [Phase 1.4] 독립 파일 저장 ===
파일 저장 완료: 'train.csv', 'val.csv', 'test.csv' (인덱스 제외)

=== [Phase 1.5] 분할 무결성 검증 (Integrity Check) ===
전체 유효 데이터 개수: 5145
Train Set: 4116개 (80.0%)
Val Set  : 514개 (10.0%)
Test Set : 515개 (10.0%)
경고: 456개의 중복 서열이 발견되었습니다! (동일한 서열이 여러 개의 라벨을 가지고 있는지 원본 데이터를 확인하세요)
